# Метаданные регионов

Собирает `metadata_train.csv` и `metadata_test.csv`. Регион кадра берётся из `dxa_utils.classify_region` по числу столбцов. Разметка бедра на уровне снимка в этот файл не входит: её собирает отдельный ноутбук.


## Пути

In [1]:
import os

import numpy as np
import pandas as pd
import pydicom

import dxa_utils as U

EXCEL_PATH = r"C:\Users\kiru\OneDrive\Рабочий стол\НД_для_обучения\разметка.xlsx"
DATASET_DIR = r"C:\Users\kiru\OneDrive\Рабочий стол\НД_для_обучения\Исследования"
TEST_DIR = r"C:\Users\kiru\OneDrive\Рабочий стол\Для теста\Для теста"

LABEL_COLUMNS = [
    "n",
    "study",
    "spine_lay",
    "spine_axis",
    "spine_obj",
    "rfem_lay",
    "rfem_roi",
    "lfem_lay",
    "lfem_roi",
    "total_spine",
    "total_rfem",
    "total_lfem",
    "comment",
    "c13",
    "c14",
    "c15",
    "c16",
    "c17",
    "c18",
]
NUMERIC_LABELS = [
    "spine_lay",
    "spine_axis",
    "spine_obj",
    "rfem_lay",
    "rfem_roi",
    "lfem_lay",
    "lfem_roi",
]


## Сбор DICOM и таблицы разметки

In [2]:
def collect_dicom_records(dataset_dir):
    rows = []
    for root, _, files in os.walk(dataset_dir):
        for fname in files:
            if not fname.lower().endswith(".dcm"):
                continue
            path = os.path.join(root, fname)
            try:
                dcm = pydicom.dcmread(path, stop_before_pixels=True)
            except Exception as exc:
                print("read error", path, exc)
                continue
            study_id = os.path.relpath(path, dataset_dir).split(os.sep)[0]
            rows.append(
                {
                    "file_path": path,
                    "file_name": fname,
                    "study_id": study_id,
                    "rows": int(getattr(dcm, "Rows", -1)),
                    "columns": int(getattr(dcm, "Columns", -1)),
                    "series_description": str(getattr(dcm, "SeriesDescription", "")),
                    "instance_number": getattr(dcm, "InstanceNumber", None),
                }
            )
    return pd.DataFrame(rows)


labels = pd.read_excel(EXCEL_PATH, header=None).iloc[2:].reset_index(drop=True)
labels.columns = LABEL_COLUMNS
labels["study"] = labels["study"].astype(str).str.strip()
for column in NUMERIC_LABELS:
    labels[column] = pd.to_numeric(labels[column], errors="coerce")
labels["has_spine"] = labels["spine_lay"].notna()
labels["has_rfem"] = labels["rfem_lay"].notna()
labels["has_lfem"] = labels["lfem_lay"].notna()


## Train и test

Из исследования `2.25.12798473087614376830819854616447908612` выкидываются кадры без уверенного региона: это битые кадры 248 px без бедренной разметки. До этого исключения в train 499 файлов, после него 497.


In [3]:
def global_quality(row):
    if row["anatomical_region"] == U.SPINE_REGION:
        if not row["has_spine"]:
            return np.nan
        return int(any(row[column] == 1 for column in ["spine_lay", "spine_axis", "spine_obj"]))
    if not (row["has_rfem"] or row["has_lfem"]):
        return np.nan
    return int(
        any(row[column] == 1 for column in ["rfem_lay", "lfem_lay", "rfem_roi", "lfem_roi"])
    )


files = collect_dicom_records(DATASET_DIR)
files[["anatomical_region", "region_confident"]] = files["columns"].apply(
    lambda value: pd.Series(U.classify_region(value))
)
merged = files.merge(labels.rename(columns={"study": "study_id"}), on="study_id", how="left")
merged["quality_class"] = merged.apply(global_quality, axis=1)
merged["label_conflict"] = (
    (merged["anatomical_region"] == U.SPINE_REGION) & ~merged["has_spine"]
) | (
    (merged["anatomical_region"] == U.FEMUR_REGION)
    & ~merged["has_rfem"]
    & ~merged["has_lfem"]
)

bad_study = "2.25.12798473087614376830819854616447908612"
bad_mask = (merged["study_id"] == bad_study) & ~merged["region_confident"]
merged = merged.loc[~bad_mask].reset_index(drop=True)

meta = merged[
    [
        "file_path",
        "file_name",
        "study_id",
        "anatomical_region",
        "region_confident",
        "quality_class",
        "label_conflict",
    ]
].copy()
meta.to_csv("metadata_train.csv", index=False)
print("saved metadata_train.csv:", len(meta))
print("region_confident=False:", int((~meta["region_confident"]).sum()))
print("label_conflict=True:", int(meta["label_conflict"].sum()))

test = collect_dicom_records(TEST_DIR)
test[["anatomical_region", "region_confident"]] = test["columns"].apply(
    lambda value: pd.Series(U.classify_region(value))
)
test.to_csv("metadata_test.csv", index=False)
print("saved metadata_test.csv:", len(test))


saved metadata_train.csv: 497
region_confident=False: 0
label_conflict=True: 0
saved metadata_test.csv: 3
